# **Plongeon Testing NB**

In [7]:
from transformers import pipeline

## Heel detection with Sapiens2 pose
Detect left/right heel positions with `facebook/sapiens2-pose-0.4b`.

In [8]:
from pathlib import Path

import torch
from PIL import Image, ImageDraw, ImageOps
from transformers import AutoImageProcessor, Sapiens2ForPoseEstimation

MODEL_ID = "facebook/sapiens2-pose-0.4b"
# Sapiens2 Goliath 308-keypoint ordering (keypoint_info of the "goliath" table in
# sapiens/pose/configs/_base_/keypoints308.py; NOT the coco_wholebody table at the top)
KEYPOINTS = {"left_ankle": 13, "right_ankle": 14, "left_heel": 17, "right_heel": 20}

img_path = Path("pictures/hurdle_step.png")
# Extra clockwise rotation (degrees) after EXIF orientation is applied, if the person still is not upright.
rotate_cw = 0

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"

In [9]:
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = Sapiens2ForPoseEstimation.from_pretrained(MODEL_ID).to(device).eval()

ImportError: 
AutoImageProcessor requires the PIL library but it was not found in your environment. You can install it with pip:
`pip install pillow`. Please note that you may need to restart your runtime after installation.


In [ ]:
orig = ImageOps.exif_transpose(Image.open(img_path)).convert("RGB")
W, H = orig.size
upright = orig.rotate(-rotate_cw, expand=True)
uw, uh = upright.size

# Top-down model: person fills the frame, so use the whole image as the person box (COCO xywh).
boxes = [[[0.0, 0.0, float(uw), float(uh)]]]
inputs = processor(images=upright, boxes=boxes, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)
person = processor.post_process_pose_estimation(outputs, boxes=boxes)[0][0]
kps, scores = person["keypoints"].cpu(), person["scores"].cpu()


def to_original(x: float, y: float) -> tuple[float, float]:
    """Map a point in the upright image back to the original (unrotated) image."""
    k = rotate_cw % 360
    if k == 0:
        return x, y
    if k == 90:   # upright = orig rotated CW 90: (x, y) -> orig (y, H-1-x)
        return y, H - 1 - x
    if k == 180:
        return W - 1 - x, H - 1 - y
    if k == 270:
        return W - 1 - y, x
    raise ValueError("rotation must be a multiple of 90")


results = {}
for name, idx in KEYPOINTS.items():
    x, y = to_original(*kps[idx].tolist())
    results[name] = (x, y, float(scores[idx]))

print(f"Image {img_path} ({W}x{H}), model {MODEL_ID} on {device}")
for name, (x, y, s) in results.items():
    print(f"  {name:12s} x={x:8.1f}  y={y:8.1f}  score={s:.3f}")

In [ ]:
# Visualization on the original image
vis = orig.copy()
d = ImageDraw.Draw(vis)
r = max(W, H) // 150
colors = {"left_heel": (0, 255, 0), "right_heel": (255, 128, 0),
          "left_ankle": (0, 120, 255), "right_ankle": (255, 0, 200)}
for name, (x, y, _) in results.items():
    d.ellipse([x - r, y - r, x + r, y + r], outline=colors[name], width=r // 2)
    d.text((x + 1.5 * r, y - r), name, fill=colors[name], font_size=r * 2)
out = img_path.with_name(img_path.stem + "_heels.png")
vis.save(out)
print(f"Saved visualization to {out}")
vis.resize((W // 6, H // 6))